## Concept primer: SQL basics

**Order SQL actually runs in** (not the order you write it):
`FROM / JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`

That explains the classic rules:
- **WHERE filters rows before grouping. HAVING filters groups after.** So `COUNT(*) > 2` must go in `HAVING`, not `WHERE`.
- A column alias made in `SELECT` usually **can't** be used in `WHERE` (WHERE runs first). SQLite is lenient, but don't rely on it.

**Joins**
- `JOIN` (inner): only rows that match in both tables.
- `LEFT JOIN`: every row from the left table; NULLs where the right has no match.
- **"Find rows with no match"** → `LEFT JOIN ... WHERE right.id IS NULL`.

**NULLs**
- `= NULL` never works; use `IS NULL` / `IS NOT NULL`.
- `COUNT(*)` counts rows; `COUNT(col)` skips NULLs.
- `COALESCE(col, 'Unknown')` → replace NULL with a value.

**Skeleton**
```sql
SELECT   c.region, COUNT(*) AS n
FROM     customers c
JOIN     accounts a ON a.customer_id = c.customer_id
WHERE    a.status = 'open'
GROUP BY c.region
HAVING   COUNT(*) > 10
ORDER BY n DESC;
```

## Problem: SQL warm-up on the bank database

Tables: `customers`, `accounts`, `transactions`, `loans` (schema prints when you run the setup cell).
Write each query as a string, look at it with `run(...)`, then `check(...)`.

**Order matters for the check: use exactly the `ORDER BY` asked for.** Target: **15 minutes**.

In [1]:
import sqlite3, hashlib
import pandas as pd
con = sqlite3.connect("../data/bank.db")

def run(sql):
    """Run a query and show the result as a table."""
    return pd.read_sql(sql, con)

_ANS = {"q1": "720e4e8fc2", "q2": "0f5007d626", "q3": "49e9f2fb61"}
def check(key, sql):
    """Checks your query's rows (values AND order) against the hidden answer."""
    rows = con.execute(sql).fetchall()
    ok = hashlib.md5(repr([tuple(r) for r in rows]).encode()).hexdigest()[:10] == _ANS[key]
    print(f"{key}: {'CORRECT' if ok else 'WRONG'}  ({len(rows)} rows)")

# Schema: look at this
for (s,) in con.execute("SELECT sql FROM sqlite_master WHERE type='table'"):
    print(s)
    print()


CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    first_name    TEXT,
    last_name     TEXT,
    birth_date    TEXT,
    city          TEXT,
    region        TEXT,
    segment       TEXT,          -- Retail / Wealth / SME  (some NULL)
    annual_income REAL,          -- some NULL
    signup_date   TEXT
)

CREATE TABLE accounts (
    account_id   INTEGER PRIMARY KEY,
    customer_id  INTEGER REFERENCES customers(customer_id),
    account_type TEXT,           -- current / savings / credit_card
    open_date    TEXT,
    close_date   TEXT,           -- NULL if still open
    balance      REAL,
    status       TEXT            -- open / closed / dormant
)

CREATE TABLE transactions (
    txn_id            INTEGER PRIMARY KEY,
    account_id        INTEGER REFERENCES accounts(account_id),
    txn_ts            TEXT,      -- 'YYYY-MM-DD HH:MM:SS'
    amount            REAL,      -- negative = debit, positive = credit
    merchant_category TEXT,      -- some NULL
    chann

## Schema map

```
customers (200 rows)                 one row = one customer
  customer_id  PK ─────────┬──────────────────────┐
  first_name, last_name    │                      │
  segment (NULLs), region  │                      │
  annual_income (NULLs)    │                      │
                           │ 1 : many             │ 1 : many
                           ▼                      ▼
accounts (372)                          loans (158)
  account_id   PK                         loan_id      PK      one row = one loan
  customer_id  FK → customers             customer_id  FK → customers
  account_type, balance, status           loan_type, principal, status, credit_score
       │ 1 : many
       ▼
transactions (9,867)                    one row = one transaction
  txn_id       PK
  account_id   FK → accounts
  amount (neg = debit), channel, is_fraud
```

Not every customer has an account (10 don't) or a loan, so start `FROM customers` when the question is about customers.
To list one table's columns: `run("PRAGMA table_info(loans)")`.

**Q1.** How many customers are in each `segment`? Show `NULL` segments as `'Unknown'`.
Columns: `segment, n_customers`. Order by `n_customers` descending, then `segment` ascending.

In [2]:
q1 = """
SELECT COALESCE(segment, 'Unknown') AS segment, COUNT(*) AS n_customers
FROM customers
GROUP BY COALESCE(segment, 'Unknown')
ORDER BY n_customers DESC, segment
"""
run(q1)

,segment,n_customers
0,Retail,124
1,SME,31
2,Unknown,24
3,Wealth,21


In [3]:
check("q1", q1)


q1: CORRECT  (4 rows)


**Q2.** Which customers have **more than 2** accounts?
Columns: `customer_id, first_name, n_accounts`. Order by `n_accounts` descending, then `customer_id` ascending.

In [4]:
q2 = """
SELECT c.customer_id, c.first_name, COUNT(*) AS n_accounts
FROM customers c
JOIN accounts a ON a.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name
HAVING COUNT(*) > 2
ORDER BY n_accounts DESC, c.customer_id
"""
run(q2)

,customer_id,first_name,n_accounts
0,3,Hugo,3
1,10,Tom,3
2,11,Isla,3
3,15,Jack,3
4,18,Hugo,3
5,19,Sara,3
6,20,Zoe,3
7,21,Rhys,3
8,22,Hugo,3
9,23,Wendy,3


In [5]:
check("q2", q2)


q2: CORRECT  (59 rows)


**Q3.** How many customers have **no loans at all**? Return a single number in a column `n_no_loan`.

In [6]:
q3 = """
SELECT COUNT(*) AS n_no_loan
FROM customers c
LEFT JOIN loans l ON l.customer_id = c.customer_id
WHERE l.loan_id IS NULL
"""
run(q3)

,n_no_loan
0,95


In [7]:
check("q3", q3)


q3: CORRECT  (1 rows)


## Explain it

1. In Q2, why does `COUNT(*) > 2` go in `HAVING` and not `WHERE`?
2. In Q3, what would go wrong if you used an inner `JOIN` instead of `LEFT JOIN`?

✅ **Model answer**

1. `WHERE` filters **rows before** grouping, when the count doesn't exist yet. `HAVING` filters **groups after** `GROUP BY` has computed `COUNT(*)`.
2. An inner `JOIN` keeps only customers who **have** a loan, so the customers with no loans, the very ones you want to count, disappear and the answer is 0. `LEFT JOIN` keeps every customer, with NULLs in the loan columns where there's no match; `WHERE l.loan_id IS NULL` then picks exactly those.